# 📚 Lumina — AI Study Assistant

## 🧠 Track B — Retrieval-Augmented Generation (RAG) System

---

## 📌 Project Overview

Lumina is an AI-powered academic assistant that allows students to upload PDF study materials and ask context-based questions.

The system uses Retrieval-Augmented Generation (RAG) to generate grounded answers directly from uploaded documents instead of relying only on the language model’s internal knowledge.

To improve factual accuracy and reduce hallucinations, Lumina retrieves the most relevant sections from the uploaded PDF using semantic similarity search before generating a response.

If the answer is not available in the uploaded material, the system refuses to generate unsupported information.

---

## ⚙️ Key Features

* 📄 PDF Upload and Processing
* ✂️ Overlapping Text Chunking
* 🔢 Embedding Generation using Sentence Transformers
* 🔎 Semantic Search using FAISS Vector Database
* 🧠 Context-Aware Response Generation using Google Gemini
* 🚫 Hallucination Control using Grounded Retrieval
* 📌 Refusal Mechanism for Out-of-Context Questions
* 💻 Interactive Colab-Based Chatbot Interface

---

## 🧪 Technical Stack

* Python
* Google Gemini API
* FAISS (Vector Similarity Search)
* Sentence Transformers
* PyPDF
* NumPy
* Google Colab

---

## 🧠 System Architecture

```text
PDF Upload
    │
    ▼
Text Extraction (PyPDF)
    │
    ▼
Chunking (500-character chunks with 100 overlap)
    │
    ▼
Embedding Generation (all-MiniLM-L6-v2)
    │
    ▼
FAISS Vector Index
    │
    ▼
User Query
    │
    ▼
Query Embedding
    │
    ▼
Top-K Retrieval
    │
    ▼
Gemini 2.5 Flash
    │
    ▼
Grounded Final Answer
```

---

## 🎯 Objective

The objective of Lumina is to build a grounded AI question-answering system that:

* Answers questions only from uploaded study material
* Reduces hallucinations using retrieval-based grounding
* Demonstrates practical RAG architecture
* Supports academic learning and revision

---

## 🔍 How the System Works

### Step 1 — PDF Upload

The user uploads one or more PDF documents containing study material.

### Step 2 — Text Extraction

Text is extracted from every page using the PyPDF library.

### Step 3 — Chunking

The extracted text is divided into overlapping chunks to preserve context during retrieval.

### Step 4 — Embedding Generation

Each chunk is converted into vector embeddings using the `all-MiniLM-L6-v2` Sentence Transformer model.

### Step 5 — FAISS Indexing

The embeddings are stored inside a FAISS vector database for efficient semantic similarity search.

### Step 6 — Query Retrieval

When the user asks a question, the query is embedded and compared against the FAISS index to retrieve the most relevant chunks.

### Step 7 — Grounded Answer Generation

The retrieved context is passed to Gemini 2.5 Flash along with strict instructions to answer only from the provided content.

---

## 🚫 Hallucination Prevention

Lumina includes grounding instructions that prevent the model from generating unsupported information.

If the answer is not found in the retrieved context, the system responds with:

"I could not find this in the uploaded study material."

This ensures reliable and document-based responses.

---

## 📊 Example Queries

| User Question                    | System Behavior                                        |
| -------------------------------- | ------------------------------------------------------ |
| What is Artificial Intelligence? | Retrieves relevant chunk and generates grounded answer |
| Explain Machine Learning.        | Uses retrieved study material context                  |
| What is Mars?                    | Refuses because answer is outside uploaded PDF         |

---

## 👨‍💻 Author

**Anush Kounain**

---

## 📌 Note

This project was developed as part of the Prompt Engineering Mastery Assignment under:

### Track B — RAG & Knowledge Systems

It demonstrates the implementation of a lightweight Retrieval-Augmented Generation pipeline using semantic retrieval and grounded response generation.


# 🔷 SECTION 1 — INSTALLATION

In [ ]:
!pip install -q google-generativeai
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf

ERROR: Could not find a version that satisfies the requirement sentence-transformer (from versions: none)
ERROR: No matching distribution found for sentence-transformer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 6.4 MB/s eta 0:00:00


# 🔷 SECTION 2 — IMPORTS

In [ ]:
import os
import faiss
import numpy as np
import google.generativeai as genai

from google.colab import files
from google.colab import userdata

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("All imports successful.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


All imports successful.


In [ ]:
import os
import google.generativeai as genai

# Retrieve Google Gemini API Key from Colab Secrets or environment
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except (ImportError, Exception):
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    import getpass
    GOOGLE_API_KEY = getpass.getpass("Enter your Gemini API key (or configure GOOGLE_API_KEY in Colab Secrets): ")

genai.configure(api_key=GOOGLE_API_KEY)


In [ ]:
model = genai.GenerativeModel("models/gemini-2.5-flash")

print("Gemini model loaded.")

Gemini model loaded.


# 🔷 SECTION 3 — PDF UPLOAD

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Comprehensive Overview of Artificial Intelligence.pdf to Comprehensive Overview of Artificial Intelligence.pdf


# 🔷 SECTION 4 — TEXT EXTRACTION

In [ ]:
all_text = ""

for filename in uploaded.keys():
    pdf = PdfReader(filename)

    for page in pdf.pages:
        text = page.extract_text()

        if text:
            all_text += text + "\n"

print("PDF text extracted successfully.")
print("Total characters:", len(all_text))

PDF text extracted successfully.
Total characters: 3307


# 🔷 SECTION 5 — CHUNKING

In [ ]:
chunk_size = 500
overlap = 100

chunks = []

for i in range(0, len(all_text), chunk_size - overlap):
    chunk = all_text[i:i + chunk_size]
    chunks.append(chunk)

print("Total chunks created:", len(chunks))

Total chunks created: 9


# 🔷 SECTION 6 — EMBEDDINGS

In [ ]:
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Embedding model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print("Embeddings created.")
print(chunk_embeddings.shape)

Embeddings created.
(9, 384)


# 🔷 SECTION 7 — FAISS INDEX

In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

print("FAISS index created.")

FAISS index created.


In [ ]:
faiss.write_index(index, "faiss_index.index")

print("FAISS index saved.")

FAISS index saved.


# 🔷 SECTION 8 — RETRIEVAL FUNCTION

In [ ]:
def retrieve_chunks(query, k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    retrieved = []

    for idx in indices[0]:
        if idx < len(chunks):
            retrieved.append(chunks[idx])

    return retrieved

In [ ]:
SYSTEM_PROMPT = """
You are Lumina, an AI assistant.

Answer ONLY using the given context from the PDF.

If the answer is not in the context, say:
"I could not find this in the uploaded study material."

Keep answers short, clear, and accurate.
Do not hallucinate.
"""

# 🔷 SECTION 9 — CHATBOT LOOP (uses MODEL here)

In [ ]:
import textwrap

print("RAG CHATBOT (type 'exit' to stop)\n")

while True:
    query = input("Q: ")

    if query.lower() == "exit":
        print("Session ended.")
        break

    context = "\n\n".join(retrieve_chunks(query))

    final_prompt = f"""{SYSTEM_PROMPT}

Context:
{context}

Question:
{query}
"""

    response = model.generate_content(final_prompt)

    print("\n" + "="*60)

    if response.candidates:
        output = response.text.replace("\n", " ").strip()
        print(textwrap.fill(output, width=80))
    else:
        print("No response generated.")
        if hasattr(response, "prompt_feedback"):
            print("Safety feedback:", response.prompt_feedback)

    print("="*60)


RAG CHATBOT (type 'exit' to stop)

Q: foundations of ml

Machine Learning (ML), a core application of AI, focuses on algorithms that
improve their efficiency automatically through experience. A well-defined
learning task can be characterized by three components:  *   **Task (T):** The
specific goal, such as categorizing email as spam or legitimate. *
**Performance (P):** The metric used to measure success, such as the percentage
of emails correctly classified. *   **Experience (E):** The data or practice
games used to improve, such as a database of human-labeled emails.
Q: what is ai

Artificial Intelligence (AI) is a discipline that provides machines with the
ability to automatically learn from data and past experiences to identify
patterns and make predictions with minimal human intervention.
Q: agent architecture

Agent Architecture: The ReAct Loop
Q: exit
Session ended.
